In [17]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report,roc_auc_score
import matplotlib.pyplot as plt

# Deep Learning With Tensor Flow/Keras

## Day 10 Tasks (Deep Learning)
- Load processed train/test splits.
- Compute class weights for imbalanced labels.
- Build a dense neural network with dropout + batch normalization.
- Train with early stopping and class weights.
- Evaluate on test set at threshold 0.40.
- Plot training history and save the model.

In [7]:
x_train = pd.read_csv("../data/processed/x_train.csv")
x_test = pd.read_csv("../data/processed/x_test.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

In [8]:
classes = np.array([0,1])
weights = compute_class_weight('balanced',classes=classes,y=y_train)
class_weight_dict = {0:weights[0],1:weights[1]}

In [ ]:
print(class_weight_dict)

In [ ]:
model = keras.Sequential([
    layers.Dense(64,activation='relu',input_shape=(35,)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(32,activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(1,activation='sigmoid')
])
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics = ['AUC','Recall']
)
model.summary()

## Model Architecture (Keras)
- Input: 35 features
- Hidden 1: Dense(64, ReLU) + BatchNorm + Dropout(0.30)
- Hidden 2: Dense(32, ReLU) + BatchNorm + Dropout(0.30)
- Output: Dense(1, Sigmoid)
- Optimizer: Adam | Loss: Binary Crossentropy | Metrics: AUC, Recall

In [11]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience = 3,
    restore_best_weights=True
)

## Regularization + Training Strategy
- Dropout: randomly disables 30% of neurons per step to reduce overfitting.
- Batch Normalization: stabilizes activations, speeds up convergence.
- Early Stopping: monitors validation loss; stops when it stops improving.
- Class Weights: address class imbalance by penalizing class 1 more.

Class weights used:
- Class 0: 0.59
- Class 1: 3.36 (about 5.7x higher penalty)

In [ ]:
model.fit(x_train,y_train,epochs=20,batch_size=1024,validation_split=0.2,class_weight=class_weight_dict,callbacks=[early_stopping])

In [ ]:
y_proba_nn = model.predict(x_test)
y_pred_nn = (y_proba_nn >= 0.4).astype(int)
print(classification_report(y_test,y_pred_nn))
nn_roc_auc = roc_auc_score(y_test,y_proba_nn)
print(f"ROC-AUC: {nn_roc_auc:.4f}")

## Neural Network Results (Test Set)
Threshold fixed at 0.40 for screening.
- ROC-AUC: 0.8099
- Precision (class 1): 0.28
- Recall (class 1): 0.83
- F1 (class 1): 0.42

Saved assets:
- Training history plot: ../reports/model/nn_training_history.png
- Model file: ../models/neural_network.keras

In [ ]:
history = model.fit(x_train,y_train,epochs=20,batch_size=1024,validation_split=0.2,class_weight=class_weight_dict,callbacks=[early_stopping])
plt.plot(history.history['AUC'], label='Train AUC')
plt.plot(history.history['val_AUC'], label='Val AUC')
plt.xlabel('Epoch')
plt.ylabel('AUC')
plt.title('Neural Network Training History')
plt.legend()
plt.savefig('../reports/model/nn_training_history.png', bbox_inches='tight', dpi=150)
plt.show()

## Model Comparison Snapshot (Screening Context)
| Model | ROC-AUC | Recall (t=0.40) |
| --- | --- | --- |
| Logistic Regression | 0.801 | 0.88 |
| Decision Tree | 0.802 | - |
| Random Forest | 0.807 | - |
| KNN | 0.690 | - |
| XGBoost | 0.813 | 0.85 |
| Neural Network | 0.807 | 0.83 |

Conclusion: XGBoost remains the top performer overall; the neural network is competitive but does not surpass it.

In [19]:
model.save('../models/neural_network.keras')

## Final Day 10 Documentation (Ready to Reuse)
Day 10 focused on building a deep learning baseline using Keras. A dense network (64 → 32 → 1) with ReLU activations, dropout (0.30), and batch normalization was trained using class weights and early stopping. Class weights were 0.59 for class 0 and 3.36 for class 1, emphasizing recall for the minority class.

The model achieved ROC-AUC 0.807 with recall 0.83 at threshold 0.40. Training history was saved, and the model was exported for reuse. When compared against previous ML models, XGBoost still leads (ROC-AUC 0.813, recall 0.85), while the neural network is a respectable but secondary candidate.

### Note on Neural Network After Feature Removal
PNEUVAC3 was removed post-SHAP analysis (reverse causality confirmed).
The neural network was not retrained after this removal for two reasons:
1. XGBoost remained the superior model even after retraining (ROC-AUC 0.808)
2. Neural networks show marginal gains on tabular data vs gradient boosting
   — retraining would not change the final model selection decision.
The NN result (ROC-AUC 0.807 with 35 features) serves as a valid 
comparison benchmark for the original feature set.